In [1]:
import pandas as pd
from models import crime_serializer, crime_from_row
from kafka import KafkaProducer

In [2]:
path = "../data/crimes_2025.csv"
columns = ['Date', 'Primary Type', 'Community Area', 'Latitude', 'Longitude']
df = pd.read_csv(path, usecols=columns).head(1000)
df = df.rename(columns={
    'Date': 'event_time',
    'Primary Type': 'crime_type',
    'Community Area': 'community_area',
    'Latitude': 'latitude',
    'Longitude': 'longitude'
})

df['event_time'] = pd.to_datetime(df['event_time'])
df.head()

/var/folders/qb/t_syfdkd58gckhlhb0bvdvm00000gn/T/ipykernel_197/3106362531.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['event_time'] = pd.to_datetime(df['event_time'])


,event_time,crime_type,community_area,latitude,longitude
0,2025-01-01,OFFENSE INVOLVING CHILDREN,69.0,41.763175,-87.611573
1,2025-01-01,BATTERY,28.0,41.873220,-87.675869
2,2025-01-01,DECEPTIVE PRACTICE,28.0,41.879709,-87.664371
3,2025-01-01,DECEPTIVE PRACTICE,25.0,41.906700,-87.757124
4,2025-01-01,DECEPTIVE PRACTICE,65.0,41.779711,-87.713851


In [3]:
# Example usage: Convert the first row to a Crime instance
crime = crime_from_row(df.iloc[0])
print(crime)

Crime(event_time=1735689600000, crime_type='OFFENSE INVOLVING CHILDREN', community_area=69, latitude=41.763175215, longitude=-87.611572995)


In [4]:
server = 'localhost:9092'
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=crime_serializer
)

topic_name = 'chi_crimes'
producer.send(topic_name, value=crime) # Directly converting the dataclass to JSON bytes
producer.flush()

In [5]:
# Send all crimes to Kafka topic in a loop
import time

t0 = time.time()

for _, row in df.iterrows():
    crime = crime_from_row(row)
    producer.send(topic_name, value=crime)
    print(f"Sent: {crime}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: Crime(event_time=1735689600000, crime_type='OFFENSE INVOLVING CHILDREN', community_area=69, latitude=41.763175215, longitude=-87.611572995)
Sent: Crime(event_time=1735689600000, crime_type='BATTERY', community_area=28, latitude=41.873219626, longitude=-87.675868666)
Sent: Crime(event_time=1735689600000, crime_type='DECEPTIVE PRACTICE', community_area=28, latitude=41.879709233, longitude=-87.664370999)
Sent: Crime(event_time=1735689600000, crime_type='DECEPTIVE PRACTICE', community_area=25, latitude=41.906699517, longitude=-87.757123673)
Sent: Crime(event_time=1735689600000, crime_type='DECEPTIVE PRACTICE', community_area=65, latitude=41.779711108, longitude=-87.713851263)
Sent: Crime(event_time=1735689600000, crime_type='BURGLARY', community_area=19, latitude=41.937821431, longitude=-87.781053508)
Sent: Crime(event_time=1735689600000, crime_type='THEFT', community_area=25, latitude=41.878606548, longitude=-87.764804593)
Sent: Crime(event_time=1735689600000, crime_type='OTHER OFFE